# Training a Potential: how to setup data pipeline, model and trainer 

# Training a Nerual Network Potential

This section introduces how to use `MolPot` to train a nnp

## Preparing and loding data

Before we can start training neural networks with `MolPot`, we need to prepare our data. 

In [1]:
%load_ext autoreload
%autoreload 2

import logging, sys
molpot_logger = logging.getLogger('molpot')
molpot_logger.setLevel(logging.INFO)

import molpot as mpot
import torch
from molpot import alias

In [2]:
# 1. get rMD17 dataset
rmd17_ds = mpot.dataset.rMD17(
    molecule="aspirin",
    save_dir="data",
    device="cuda",
    preprocess=[
        mpot.locality.NeighborList(
            cutoff=5,
            index_dtype=torch.int64,
            exclude_ii=True,
        )
    ],
)
rmd17_ds.prepare_data()
data_inspector = mpot.inspector.DataInspector(rmd17_ds)
data_inspector.summary()

number of data: 1000

                          dataset: rmd17                           
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ label  ┃          type          ┃    unit    ┃     comment      ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ energy │    <class 'float'>     │  kcal/mol  │ potential_energy │
│ forces │ <class 'torch.Tensor'> │ kcal/mol/A │      forces      │
└────────┴────────────────────────┴────────────┴──────────────────┘

In [3]:
train_ds, valid_ds = torch.utils.data.random_split(rmd17_ds, [.8, .2])

train_dl = mpot.DataLoader(
    dataset=train_ds,
    batch_size=10,
    shuffle=False,
)
valid_dl = mpot.DataLoader(
    dataset=valid_ds,
    batch_size=10,
    shuffle=False,
)

In [4]:
for frame in train_dl:
    print(frame)
    break

Frame(
    fields={
        angles: TensorDict(
            fields={
            },
            batch_size=torch.Size([]),
            device=None,
            is_shared=False),
        atoms: TensorDict(
            fields={
                Z: Tensor(shape=torch.Size([210]), device=cpu, dtype=torch.int32, is_shared=False),
                atomic_batch_mask: Tensor(shape=torch.Size([210]), device=cpu, dtype=torch.int64, is_shared=False),
                xyz: Tensor(shape=torch.Size([210, 3]), device=cpu, dtype=torch.float32, is_shared=False)},
            batch_size=torch.Size([]),
            device=None,
            is_shared=False),
        bonds: TensorDict(
            fields={
            },
            batch_size=torch.Size([]),
            device=None,
            is_shared=False),
        box: TensorDict(
            fields={
            },
            batch_size=torch.Size([]),
            device=None,
            is_shared=False),
        cell: TensorDict(
            fields

[W911 12:21:28.706036183 LinearAlgebra.cpp:3072] Warning: at::frobenius_norm is deprecated and it is just left for JIT compatibility. It will be removed in a future PyTorch release. Please use `linalg.vector_norm(A, 2., dim, keepdim)` instead (function operator())
/opt/conda/lib/python3.12/site-packages/torch/nested/__init__.py:220: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  return _nested.nested_tensor(


## Setting up the model

In [5]:
cutoff = 5.0
n_rbf = 10
arch = mpot.nnp.PiNet(
    depth=4,
    basis_fn=mpot.nnp.GaussianRBF(n_rbf, cutoff),
    cutoff_fn=mpot.nnp.CosineCutoff(cutoff),
    pp_nodes=[16],
    pi_nodes=[16],
    ii_nodes=[16],
    max_atomtypes=10,
)
energy_readout = mpot.nnp.Atomwise(16, 1, [], from_key=("pinet", "p1"), to_key=("predicts", "energy"), aggregation_mode="add")

model = mpot.PotentialSeq("pinet", arch, energy_readout)

In [13]:
import torch

loss_fn = mpot.engine.loss.loss_wrapper(
    input_=("predicts", "energy"), target=("props", "energy")
)(torch.nn.MSELoss())
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, 0.994)

trainer = mpot.PotentialTrainer(
    "pinet-rmd17",
    model,
    loss_fn,
    optimizer,
    scheduler,
    root_dir="runs",
    device="cuda"
)

profiler = mpot.engine.fix.tensorboard.Profiler(wait=1, warmup=1, active=3, log_dir="runs/pinet-rmd17/profiler")

trainer.fix.register(profiler.before_epoch, trainer.Stage.before_epoch)
trainer.fix.register(profiler.after_epoch, trainer.Stage.after_epoch)
trainer.fix.register(profiler.after_step, trainer.Stage.after_step)

trainer.fix.register(
    mpot.engine.fix.TensorBoardFix(
        every_n_steps=1,
        log_dir="runs/pinet-rmd17",
    ),
    trainer.Stage.after_step,
)
trainer.fix.register(
    mpot.engine.fix.EpochSpeed(every_n_epochs=1), trainer.Stage.after_epoch
)
trainer.fix.register(
    mpot.engine.fix.StepSpeed(every_n_steps=100), trainer.Stage.after_step
)
trainer.fix.register(
    mpot.engine.fix.MAE(
        output_key="e_mae",
        every_n_steps=100,
        result_key=("predicts", "energy"),
        target_key=("props", "energy"),
    ),
    trainer.Stage.after_step,
)
# TODO: ckpt, valid
trainer.fix.register(mpot.engine.fix.Validation(1, valid_dl), trainer.Stage.after_epoch)

In [15]:
%load_ext tensorboard
# trainer.compile()
inputs = trainer.train(train_dl, 1000000)

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
0
80
160
240
320
400
480
560
640


# Training a Classic Potential

This section introduces how to use `MolPot` to train a classic forcefield

In [8]:
import molpy as mp

ModuleNotFoundError: No module named 'molpy'

In [ ]:
ff = mp.ForceField("gaff")

atomstyle = ff.def_atomstyle("full")
C = atomstyle.def_atomtype("C", 1, {"mass": 12.01})
H = atomstyle.def_atomtype("H", 2, {"mass": 1.01})
O = atomstyle.def_atomtype("O", 3, {"mass": 16.00})
N = atomstyle.def_atomtype("N", 4, {"mass": 14.01})

bondstyle = ff.def_bondstyle("harmonic")